In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType, FloatType, DoubleType

# Connect with Spark

In [2]:
spark = SparkSession\
    .builder\
    .master("local[*]")\
    .appName("Iceberg")\
    .config("spark.log.level", "WARN")\
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")\
    .config("spark.sql.defaultCatalog", "lakekeeper")\
    .config("spark.sql.catalog.lakekeeper", "org.apache.iceberg.spark.SparkCatalog")\
    .config("spark.sql.catalog.lakekeeper.type", "rest")\
    .config("spark.sql.catalog.lakekeeper.uri", "http://lakekeeper:9191/catalog")\
    .config("spark.sql.catalog.lakekeeper.warehouse", "demo")\
    .config("spark.sql.catalog.lakekeeper.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")\
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.10.1,org.apache.iceberg:iceberg-aws-bundle:1.10.1")\
    .getOrCreate()


:: loading settings :: url = jar:file:/opt/spark-3.5.8-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vadim/.ivy2/cache
The jars for the packages stored in: /home/vadim/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e6e4cf71-a6d1-4c02-ba1f-f766ebfb184b;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.10.1 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.10.1 in central
:: resolution report :: resolve 108ms :: artifacts dl 3ms
	:: modules in use:
	org.apache.iceberg#iceberg-aws-bundle;1.10.1 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.10.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-----------------------------------------------

In [3]:
print(spark.version)

3.5.8


# Создаём namespace

In [4]:
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS spark_demo")
spark.sql("SHOW NAMESPACES").show()

+--------------+
|     namespace|
+--------------+
|pyiceberg_demo|
|    spark_demo|
+--------------+



# Создаём таблицу

In [5]:
spark.sql("DROP TABLE IF EXISTS spark_demo.taxis")

DataFrame[]

In [6]:
schema = StructType([
    StructField("vendor_id", LongType(), True),
    StructField("trip_id", LongType(), True),
    StructField("trip_distance", FloatType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True)
])

In [7]:
df = spark.createDataFrame([], schema)
df.writeTo("spark_demo.taxis").create()

In [8]:
spark.table("spark_demo.taxis").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
+---------+-------+-------------+-----------+------------------+



In [9]:
spark.sql("ALTER TABLE spark_demo.taxis CREATE TAG `initial`")

DataFrame[]

# Writing Data to a Table

In [10]:
import time

time.sleep(600)

In [11]:
data1 = [
    [1, 1000371, 1.8, 15.32, "N"],
    [2, 1000372, 2.5, 22.15, "N"],
    [2, 1000373, 0.9, 9.01, "N"],
    [1, 1000374, 8.4, 42.13, "Y"]
]

df1 = spark.createDataFrame(data1, schema)

In [12]:
df1.writeTo("spark_demo.taxis").append()

In [13]:
spark.table("spark_demo.taxis").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
|        1|1000371|          1.8|      15.32|                 N|
|        2|1000372|          2.5|      22.15|                 N|
|        2|1000373|          0.9|       9.01|                 N|
|        1|1000374|          8.4|      42.13|                 Y|
+---------+-------+-------------+-----------+------------------+



In [14]:
spark.sql("ALTER TABLE spark_demo.taxis CREATE TAG `first_write`")

DataFrame[]

In [15]:
time.sleep(600)

In [16]:
data2 = [
    [3, 2000371, 1.8, 15.32, "N"],
    [4, 2000372, 2.5, 22.15, "N"],
    [4, 2000373, 0.9, 9.01, "N"],
    [3, 2000374, 8.4, 42.13, "Y"]
]

df2 = spark.createDataFrame(data2, schema)
df2.writeTo("spark_demo.taxis").append()

In [17]:
spark.table("spark_demo.taxis").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
|        1|1000371|          1.8|      15.32|                 N|
|        3|2000371|          1.8|      15.32|                 N|
|        2|1000372|          2.5|      22.15|                 N|
|        4|2000372|          2.5|      22.15|                 N|
|        4|2000373|          0.9|       9.01|                 N|
|        2|1000373|          0.9|       9.01|                 N|
|        1|1000374|          8.4|      42.13|                 Y|
|        3|2000374|          8.4|      42.13|                 Y|
+---------+-------+-------------+-----------+------------------+



In [18]:
spark.sql("ALTER TABLE spark_demo.taxis CREATE TAG `second_write`")

DataFrame[]

In [19]:
spark.sql("SELECT * FROM spark_demo.taxis.refs").show()

+------------+------+-------------------+-----------------------+---------------------+----------------------+
|        name|  type|        snapshot_id|max_reference_age_in_ms|min_snapshots_to_keep|max_snapshot_age_in_ms|
+------------+------+-------------------+-----------------------+---------------------+----------------------+
| first_write|   TAG|2560223144540635088|                   NULL|                 NULL|                  NULL|
|        main|BRANCH|3347609563702847672|                   NULL|                 NULL|                  NULL|
|second_write|   TAG|3347609563702847672|                   NULL|                 NULL|                  NULL|
|     initial|   TAG| 615738162196438231|                   NULL|                 NULL|                  NULL|
+------------+------+-------------------+-----------------------+---------------------+----------------------+



# Snapshots

In [20]:
spark.sql("SELECT * FROM spark_demo.taxis.snapshots ORDER BY committed_at").show()

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2026-07-03 11:13:...| 615738162196438231|               NULL|   append|s3://lakehouse/01...|{changed-partitio...|
|2026-07-03 11:23:...|2560223144540635088| 615738162196438231|   append|s3://lakehouse/01...|{total-data-files...|
|2026-07-03 11:33:...|3347609563702847672|2560223144540635088|   append|s3://lakehouse/01...|{app-id -> local-...|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+



In [21]:
spark.sql("""SELECT
    committed_at,
    snapshot_id,
    parent_id
FROM spark_demo.taxis.snapshots
ORDER BY committed_at""").show(10, False)

+-----------------------+-------------------+-------------------+
|committed_at           |snapshot_id        |parent_id          |
+-----------------------+-------------------+-------------------+
|2026-07-03 11:13:39.85 |615738162196438231 |NULL               |
|2026-07-03 11:23:42.869|2560223144540635088|615738162196438231 |
|2026-07-03 11:33:45.439|3347609563702847672|2560223144540635088|
+-----------------------+-------------------+-------------------+



In [22]:
ts = spark.sql("""SELECT
    date_trunc('MINUTE', committed_at + INTERVAL '5 MINUTES'),
    snapshot_id
FROM spark_demo.taxis.snapshots
ORDER BY committed_at""").collect()

In [23]:
ts1 = ts[0][0].strftime("%Y-%m-%d %H:%M:%S")
ts2 = ts[1][0].strftime("%Y-%m-%d %H:%M:%S")
ts3 = ts[2][0].strftime("%Y-%m-%d %H:%M:%S")

In [24]:
snap1 = ts[0][1]
snap2 = ts[1][1]
snap3 = ts[2][1]

In [25]:
spark.sql(f"CALL lakekeeper.system.rollback_to_snapshot('spark_demo.taxis', {snap1})")

DataFrame[previous_snapshot_id: bigint, current_snapshot_id: bigint]

In [26]:
spark.table("spark_demo.taxis").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
+---------+-------+-------------+-----------+------------------+



In [27]:
spark.sql("SELECT * FROM spark_demo.taxis.snapshots ORDER BY committed_at").show()

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2026-07-03 11:13:...| 615738162196438231|               NULL|   append|s3://lakehouse/01...|{total-equality-d...|
|2026-07-03 11:23:...|2560223144540635088| 615738162196438231|   append|s3://lakehouse/01...|{engine-name -> s...|
|2026-07-03 11:33:...|3347609563702847672|2560223144540635088|   append|s3://lakehouse/01...|{added-files-size...|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+



In [39]:
spark.sql(f"CALL lakekeeper.system.rollback_to_timestamp('spark_demo.taxis', TIMESTAMP '{ts2}')")

DataFrame[previous_snapshot_id: bigint, current_snapshot_id: bigint]

In [40]:
spark.table("spark_demo.taxis").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
|        1|1000371|          1.8|      15.32|                 N|
|        2|1000372|          2.5|      22.15|                 N|
|        2|1000373|          0.9|       9.01|                 N|
|        1|1000374|          8.4|      42.13|                 Y|
+---------+-------+-------------+-----------+------------------+



In [30]:
spark.sql(f"CALL lakekeeper.system.set_current_snapshot('spark_demo.taxis', {snap3})")

DataFrame[previous_snapshot_id: bigint, current_snapshot_id: bigint]

In [31]:
spark.table("spark_demo.taxis").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
|        3|2000371|          1.8|      15.32|                 N|
|        4|2000372|          2.5|      22.15|                 N|
|        4|2000373|          0.9|       9.01|                 N|
|        3|2000374|          8.4|      42.13|                 Y|
|        1|1000371|          1.8|      15.32|                 N|
|        2|1000372|          2.5|      22.15|                 N|
|        2|1000373|          0.9|       9.01|                 N|
|        1|1000374|          8.4|      42.13|                 Y|
+---------+-------+-------------+-----------+------------------+



# Inspecting tables

In [32]:
spark.sql("SELECT * FROM spark_demo.taxis.history ORDER BY made_current_at").show(10, False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-07-03 11:13:39.85 |615738162196438231 |NULL               |true               |
|2026-07-03 11:23:42.869|2560223144540635088|615738162196438231 |true               |
|2026-07-03 11:33:45.439|3347609563702847672|2560223144540635088|true               |
|2026-07-03 11:33:47.525|615738162196438231 |NULL               |true               |
|2026-07-03 11:35:48.113|3347609563702847672|2560223144540635088|true               |
+-----------------------+-------------------+-------------------+-------------------+



# Time travel

In [33]:
spark.sql(f"SELECT * FROM spark_demo.taxis TIMESTAMP AS OF '{ts2}'").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
|        1|1000371|          1.8|      15.32|                 N|
|        2|1000372|          2.5|      22.15|                 N|
|        2|1000373|          0.9|       9.01|                 N|
|        1|1000374|          8.4|      42.13|                 Y|
+---------+-------+-------------+-----------+------------------+



In [34]:
spark.sql(f"SELECT * FROM spark_demo.taxis VERSION AS OF {snap1}").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
+---------+-------+-------------+-----------+------------------+



In [35]:
spark.sql("SELECT * FROM spark_demo.taxis VERSION AS OF 'first_write'").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
|        1|1000371|          1.8|      15.32|                 N|
|        2|1000372|          2.5|      22.15|                 N|
|        2|1000373|          0.9|       9.01|                 N|
|        1|1000374|          8.4|      42.13|                 Y|
+---------+-------+-------------+-----------+------------------+



Metadata tables can also be inspected with time travel using the DataFrameReader API

In [36]:
taxis = spark\
            .read\
            .format("iceberg")\
            .option("snapshot-id", snap2)\
            .load("spark_demo.taxis")

taxis.show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
|        1|1000371|          1.8|      15.32|                 N|
|        2|1000372|          2.5|      22.15|                 N|
|        2|1000373|          0.9|       9.01|                 N|
|        1|1000374|          8.4|      42.13|                 Y|
+---------+-------+-------------+-----------+------------------+



# Files

In [37]:
spark.sql("""SELECT
    content,
    file_path,
    file_format,
    spec_id,
    record_count,
    file_size_in_bytes
FROM spark_demo.taxis.files""").show()

+-------+--------------------+-----------+-------+------------+------------------+
|content|           file_path|file_format|spec_id|record_count|file_size_in_bytes|
+-------+--------------------+-----------+-------+------------+------------------+
|      0|s3://lakehouse/01...|    PARQUET|      0|           1|              1553|
|      0|s3://lakehouse/01...|    PARQUET|      0|           1|              1550|
|      0|s3://lakehouse/01...|    PARQUET|      0|           1|              1553|
|      0|s3://lakehouse/01...|    PARQUET|      0|           1|              1553|
|      0|s3://lakehouse/01...|    PARQUET|      0|           1|              1553|
|      0|s3://lakehouse/01...|    PARQUET|      0|           1|              1548|
|      0|s3://lakehouse/01...|    PARQUET|      0|           1|              1552|
|      0|s3://lakehouse/01...|    PARQUET|      0|           1|              1553|
+-------+--------------------+-----------+-------+------------+------------------+



# Table Statistics

In [38]:
spark.sql("CALL lakekeeper.system.compute_table_stats('spark_demo.taxis')").show(10, False)

+---------------------------------------------------------------------------------------------------------------------------+
|statistics_file                                                                                                            |
+---------------------------------------------------------------------------------------------------------------------------+
|s3://lakehouse/019f270a-55b6-7401-8608-b30ac1fd1b3b/metadata/3347609563702847672-8334b3c5-e537-4ed3-bc74-73607167b0a0.stats|
+---------------------------------------------------------------------------------------------------------------------------+

